# Ofsted baselines

Non-neural baselines for the 4-class Ofsted grade task, trained and evaluated on
exactly the same arrays as `ann_ofsted.ipynb` (saved by §6 of `preprocessing.ipynb`).

- Majority class and stratified random guessing, as floors
- Logistic regression, unweighted and class-balanced
- Random forest, unweighted, class-balanced, and class-balanced with shallower trees

Hyperparameters are compared on validation macro-F1; the test set is reported once
for every model so the ANN comparison is like-for-like. Test-set metrics are written
to `../results/ofsted_baseline_test_metrics.csv` for the comparison cell in the ANN notebook.

In [1]:
#Load data and set up

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             accuracy_score, balanced_accuracy_score)

SEED = 42
DATA = Path("../data")
RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

X_train = np.load(DATA / "X_cls_train.npy")
X_val   = np.load(DATA / "X_cls_val.npy")
X_test  = np.load(DATA / "X_cls_test.npy")
y_train = np.load(DATA / "y_cls_train.npy")
y_val   = np.load(DATA / "y_cls_val.npy")
y_test  = np.load(DATA / "y_cls_test.npy")

grade_labels = ["Inadequate", "Requires improvement", "Good", "Outstanding"]
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Train: (1269, 16) | Val: (272, 16) | Test: (273, 16)


In [2]:
#Fit baselines

models = {
    "Majority class":       DummyClassifier(strategy="most_frequent"),
    "Stratified random":    DummyClassifier(strategy="stratified", random_state=SEED),
    "Logistic Regression":  LogisticRegression(max_iter=2000),
    "Logistic Regression (balanced)": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest":        RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "Random Forest (balanced)": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1),
    # Shallower trees + larger leaves: less overfitting to the 38 Inadequate training schools
    "Random Forest (balanced, depth 8)": RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=5, class_weight="balanced",
        random_state=SEED, n_jobs=-1),
}

rows, test_preds = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    val_pred, test_pred = model.predict(X_val), model.predict(X_test)
    test_preds[name] = test_pred
    rows.append({
        "Model": name,
        "Val Macro-F1":  f1_score(y_val, val_pred, average="macro"),
        "Test Macro-F1": f1_score(y_test, test_pred, average="macro"),
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Test Balanced Acc": balanced_accuracy_score(y_test, test_pred),
    })

results = pd.DataFrame(rows).round(3)
print(results.to_string(index=False))

                            Model  Val Macro-F1  Test Macro-F1  Test Accuracy  Test Balanced Acc
                   Majority class         0.202          0.203          0.681              0.250
                Stratified random         0.232          0.243          0.513              0.243
              Logistic Regression         0.362          0.370          0.703              0.360
   Logistic Regression (balanced)         0.376          0.350          0.407              0.435
                    Random Forest         0.409          0.365          0.696              0.357
         Random Forest (balanced)         0.468          0.428          0.626              0.476
Random Forest (balanced, depth 8)         0.472          0.437          0.553              0.535


In [3]:
#Per-class detail for the strongest baseline

best_name = results.loc[results["Val Macro-F1"].idxmax(), "Model"]
print(f"Best on validation: {best_name}\n")
print(classification_report(y_test, test_preds[best_name], target_names=grade_labels, zero_division=0))
print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(confusion_matrix(y_test, test_preds[best_name]),
                   index=grade_labels, columns=grade_labels))

Best on validation: Random Forest (balanced, depth 8)

                      precision    recall  f1-score   support

          Inadequate       0.06      0.12      0.08         8
Requires improvement       0.30      0.66      0.41        38
                Good       0.88      0.48      0.62       186
         Outstanding       0.49      0.88      0.63        41

            accuracy                           0.55       273
           macro avg       0.43      0.53      0.44       273
        weighted avg       0.72      0.55      0.58       273

Confusion matrix (rows=true, cols=predicted):
                      Inadequate  Requires improvement  Good  Outstanding
Inadequate                     1                     6     1            0
Requires improvement           5                    25     8            0
Good                           9                    51    89           37
Outstanding                    1                     1     3           36


In [4]:
#Save test-set metrics for the ANN comparison

results.to_csv(RESULTS / "ofsted_baseline_test_metrics.csv", index=False)
print(f"Saved to {RESULTS / 'ofsted_baseline_test_metrics.csv'}")

Saved to ../results/ofsted_baseline_test_metrics.csv


**Notes.** The test set holds only 8 Inadequate schools, so macro-F1 on it moves by
several points with a single prediction. `ann_ofsted.ipynb` therefore also compares
the ANN and the random forest with repeated stratified 5-fold cross-validation.